<a href="https://colab.research.google.com/github/kgpark88/ai-summary/blob/main/10_DNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ?????(DNN, Deep Neural Network)

? ????? ???(import)  
? ??? ????(Loading the data)  
? ??? ??? ??(Exploratory Data Analysis)  
? ??? ???(Data PreProcessing) : ????? ??, Null ??? ??, ????? ??, 
???? ???, ???? ??, ?? ?? (feature engineering) ?  
? Train, Test  ???? ??  
? ??? ???(Normalizing the Data)  
? ?? ??(Creating the Model)  
? ?? ?? ??

### ? ????? ???

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation, Dropout

### ? ??? ??

In [ ]:
# df = pd.read_csv('churn_data.csv')
df = pd.read_csv('https://raw.githubusercontent.com/kgpark88/ai-summary/main/churn_data.csv')

### ? ??? ??

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.describe().transpose()

In [ ]:
df.corr()['MonthlyCharges'][:-1].sort_values().plot(kind='bar')

### ? ??? ???

In [ ]:
df.drop('customerID', axis=1, inplace=True)

In [ ]:
df['TotalCharges'].replace([' '], ['0'], inplace=True)

In [ ]:
df['TotalCharges'] = df['TotalCharges'].astype(float)

In [ ]:
df['Churn'].replace(['Yes', 'No'], [1, 0], inplace=True)

### ?? ?? ??

In [ ]:
cols = ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 
        'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 
        'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 
        'PaperlessBilling', 'PaymentMethod']

In [ ]:
dummies = pd.get_dummies(df[cols], drop_first=True)

In [ ]:
df = df.drop(cols, axis=1)

In [ ]:
df = pd.concat([df, dummies], axis=1)

In [ ]:
# df = pd.get_dummies(df)
# cols = list(df.select_dtypes('object').columns)

### ? Train, Test  ???? ??

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X = df.drop('Churn', axis=1).values
y = df['Churn'].values

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size=0.3, 
                                                    random_state=42)

In [ ]:
X_train.shape

In [ ]:
y_train.shape

### ??? ???/????(Normalizing/Scaling)

In [ ]:
from sklearn.preprocessing import MinMaxScaler

In [ ]:
scaler = MinMaxScaler()
scaler.fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

### ?? ????, ???? ??

In [ ]:
X_train.shape

In [ ]:
y_train.shape

### ? ?? ??

In [ ]:
model = Sequential()

# input layer
model.add(Dense(64, activation='relu', input_shape=(30,)))

# hidden layer
model.add(Dense(64, activation='relu'))

# hidden layer
model.add(Dense(32, activation='relu'))

# output layer
model.add(Dense(1, activation='sigmoid'))

### ?? ?? -  ??? ??

In [ ]:
model = Sequential()
model.add(Dense(128, activation='relu', input_shape=(30,)))
model.add(Dropout(0.5))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(32, activation='relu'))
model.add(Dropout(0.5))

model.add(Dense(1, activation='sigmoid'))

### ?? ??

In [ ]:
model.summary()

In [ ]:
hidden1 = model.layers[0]

In [ ]:
weights, biases = hidden1.get_weights()

In [ ]:
print(weights)

In [ ]:
print(biases)

#### ?? ??? ? ?? ?? ??

In [ ]:
model.compile(optimizer='adam', 
              loss='binary_crossentropy', 
              metrics=['accuracy']) 

- ?? ??? ? ?? ?? ??  
model.compile(optimizer='adam', 
              loss='categorical_crossentropy', 
              metrics=['accuracy']) 


- ?? ??? ? ?? ??
model.compile(optimizer='adam', loss='mse')

### ?? ??(??) ??

In [ ]:
model.fit(X_train, y_train, 
          validation_data=(X_test, y_test),
          epochs=20, 
          batch_size=10)

### ?? ??(??) - ????

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', mode='min', 
                           verbose=1, patience=25)

In [ ]:
check_point = ModelCheckpoint('best_model.h5', verbose=1,
                              monitor='val_loss', mode='min', 
                              save_best_only=True)

In [ ]:
history = model.fit(x=X_train, y=y_train, 
          epochs=50 , batch_size=20,
          validation_data=(X_test, y_test), 
          verbose=1,
          callbacks=[early_stop, check_point])

### ? ?? ?? ??

In [ ]:
losses = pd.DataFrame(model.history.history)

In [ ]:
losses.head()

In [ ]:
losses[['loss','val_loss']].plot()

In [ ]:
losses[['loss','val_loss', 'accuracy','val_accuracy']].plot()

In [ ]:
from sklearn.metrics import classification_report,confusion_matrix

In [ ]:
predictions = model.predict_classes(X_test)

In [ ]:
print(classification_report(y_test,predictions))

In [ ]:
print(confusion_matrix(y_test,predictions))